# Exécution réelle des graphes GNU Radio dans Google Colab

Ce carnet installe **GNU Radio 3.10** dans la machine virtuelle de Colab, puis exécute les graphes de flux `.grc` du dossier `GNU_Radio` (AM, BLU, FM, PM), sans matériel de radio logicielle.
Comme Colab n'a pas d'écran, les afficheurs QT GUI sont remplacés automatiquement par des *Vector Sink* et les curseurs par des variables ; les courbes sont ensuite tracées avec matplotlib.

**Mode d'emploi :** Exécution > Tout exécuter. À la cellule 2, choisir le fichier `Simulateurs_Simulink_GNURadio_AM_BLU_FM_PM.zip`.

In [ ]:
# 1. Installation de GNU Radio (2 à 4 minutes)
!apt-get -qq update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y gnuradio > /dev/null
!gnuradio-config-info --version

In [ ]:
# 2. Envoi du dossier des simulateurs
from google.colab import files
import zipfile, os, sys
up = files.upload()
zipfile.ZipFile(list(up)[0]).extractall('/content/sim')
sys.path.insert(0, '/content/sim/En_ligne')
import colab_tools as C
G = '/content/sim/GNU_Radio/'
print(sorted(f for f in os.listdir(G) if f.endswith('.grc')))

In [ ]:
# 3. Fonctions d'affichage et de mesure
import numpy as np, matplotlib.pyplot as plt
from scipy.signal.windows import blackmanharris
def tracer(data, fs, titre, n_temps=2000):
    for nom, (kind, xs) in data.items():
        plt.figure(figsize=(10, 2.6))
        for k, x in enumerate(xs):
            if kind == 'qtgui_time_sink_x':
                plt.plot(np.real(x[-n_temps:]), lw=0.8, label=f'entrée {k}'); plt.xlabel('échantillons')
            else:
                N = 8192; w = blackmanharris(N); segs = [x[i:i + N] for i in range(len(x) // 2, len(x) - N, N // 2)]
                P = np.mean([np.abs(np.fft.fftshift(np.fft.fft(s * w))) ** 2 for s in segs], 0); P = 10 * np.log10(P / P.max() + 1e-16)
                plt.plot(np.arange(-N // 2, N // 2) / N, P, lw=0.7, label=f'entrée {k}'); plt.ylim(-120, 5); plt.xlabel('fréquence normalisée f/fs')
        plt.title(f'{titre} : {nom}'); plt.grid(ls=':'); plt.legend(); plt.show()
def comparer(d):
    a, b = [np.asarray(v) for v in d['scope'][1]]; n = min(len(a), len(b)); a, b = a[n // 2:n], b[n // 2:n]
    print('Écart efficace message démodulé / message émis retardé : %.3f ; corrélation : %.4f' % (np.sqrt(np.mean((a - b) ** 2)), np.corrcoef(a, b)[0, 1]))
def enveloppe(d):
    s = np.asarray(d['scope_emis'][1][0]); s = s[len(s) // 2:]
    from scipy.signal import hilbert
    a = np.abs(hilbert(s))[500:-500]; print('Enveloppe du signal émis : de %.3f à %.3f' % (a.min(), a.max()))

In [ ]:
# 4. AM : signal émis (scope_emis), message démodulé, Repeat contre rééchantillonneur
tb, d = C.executer(G + 'am_tx_rx_sim.grc', duree=2.0, noise_amp=0.05)
enveloppe(d); comparer(d); tracer(d, 480000, 'AM')

In [ ]:
# 5. BLU (lo_phase = 0, puis 0,7 rad : la corrélation devient cos(0,7) = 0,76)
for ph in (0.0, 0.7):
    tb, d = C.executer(G + 'ssb_tx_rx_sim.grc', duree=2.0, noise_amp=0.02, lo_phase=ph)
    print('lo_phase = %.1f :' % ph, end=' '); enveloppe(d); comparer(d)
tracer(d, 480000, 'BLU')

In [ ]:
# 6. FM
tb, d = C.executer(G + 'fm_tx_rx_sim.grc', duree=2.0, noise_amp=0.05)
comparer(d); tracer(d, 480000, 'FM')

In [ ]:
# 7. PM (la dérotation rend le récepteur insensible au déphasage de l'oscillateur local)
for ph in (0.7, 2.0, -2.5):
    tb, d = C.executer(G + 'pm_tx_rx_sim.grc', duree=2.0, noise_amp=0.05, lo_phase=ph)
    print('lo_phase = %+.1f rad :' % ph, end=' '); comparer(d)
tracer(d, 480000, 'PM')

Résultats attendus (guide, tableaux 3.03, 3.05, 4.05 et 4.07) : enveloppe AM de 0,5 à 1,5 ; BLU d'enveloppe constante ; écart efficace d'environ 0,03 entre message démodulé et message émis retardé, sans bruit ; en BLU, corrélation égale à cos(lo_phase). Les courbes `scope_emis` montrent le signal modulé : enveloppe AM, fréquence FM de 5 à 15 kHz, avance et retard de phase PM de ±1,5 rad.